### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy
%pip install torch

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import time

print(f"Numpy Version: {np.__version__}")
print(f"\nTorch Version: {torch.__version__}")

### 03 - Classe Auxiliar

In [ ]:
class MLP_Classifier:
    def __init__(
            self,
            hidden_layers = [64, 32],
            activation = "relu",
            learning_rate=0.01,
            epochs=100,
            patience=5,
            batch_size=32,
            optimizer="adam",
            regularization=None,
            dropout_p=0.1,
            lambda_l2=0.001,
            random_state=None
        ):
        self.hidden_layers = hidden_layers
        self.activation = activation
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.patience = patience
        self.batch_size = batch_size
        self.optimizer = optimizer
        self.regularization = regularization
        self.dropout_p = dropout_p
        self.lambda_l2 = lambda_l2
        self.random_state = random_state
        
        self.model = None
        self.loss_history = { "train": [], "val": [] }
        self.accuracy_history = { "train": [], "val": [] }
        self.classes = None
        self.device = torch.device("cuda" if (torch.cuda.is_avaiable()) else "cpu")

        print(f"Dispositivo: {self.device}")

        print(f"\nCuda: {"Habilitado" if (self.device == "cuda") else "Desabilitado"}")

        torch.cuda.empty_cache()

        if random_state is not None:
            torch.manual_sedd(random_state)

            np.random.seed(random_state)

    def _build_model(self, input_dim, output_dim):
        layers = []

        prevision_dim = input_dim

        for dim in self.hidden_layers:
            layers.append(nn.Linear(prevision_dim, dim))

            if (self.activation == "relu"):
                layers.append(nn.ReLU())
            elif (self.activation == "tanh"):
                layers.append(nn.Tanh())

            if (self.regularization == "dropout"):
                layers.append(nn.Dropout(p=self.dropout_p))

            prevision_dim = dim

        layers.append(nn.Linear(prevision_dim, dim))

        return nn.Sequential(* layers).to(self.device)

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        start = time.time()

        X_train_tensor = torch.FloatTensor(X_train).to(self.device)

        y_train_tensor = torch.LongTensor(y_train).to(self.device)

        if (X_val is not None and y_val is not None):
            X_val_tensor = torch.FloatTensor(X_val).to(self.device)

            y_val_tensor = torch.LongTensor(y_val).to(self.device)

            validation_data = (X_val_tensor, y_val_tensor)
        else:
            validation_data = None

        input_dim = X_train.shape[1]

        self.classes = torch.unique(y_train_tensor)

        output_dim = len(self.classes_)

        self.model = self._build_model(input_dim, output_dim)